# TIDEST quick-start (synthetic)

**TIDEST** (Testing Imputed Differential Expressions for Spatial Transcriptomics) estimates the causal effect of a binary spatial treatment on gene expression using a Robinson partially-linear model, with a Pearson-corrected *augmented outcome* built from imputed expression.

This notebook runs the full pipeline on a small synthetic dataset in a few seconds. **No data download and no R / SpatialPCA install** are required: a precomputed spatial-PC confounder matrix `U` is supplied to `fit`, bypassing the R step.

Run from the repository root so that `examples/_synthetic.py` is importable.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'examples') if os.path.isdir('examples') else os.getcwd())

import numpy as np
from tidest import tidest
from _synthetic import make_dataset

## 1. Build a synthetic dataset

300 spots, 30 true DE genes + 70 null genes, organised into 10 gene modules. We get back the single-cell reference, spatial transcriptomics counts, imputed expression, a treatment vector, and a precomputed spatial-PC matrix `U`.

In [ ]:
data = make_dataset(N=300, G_DE=30, G_null=70, M=10, n_pcs=20, seed=0)
print('ST:', data['st_adata'].shape, '| pred:', data['pred_adata'].shape, '| U:', data['U'].shape)

## 2. Fit TIDEST

- `sc_adata` &rarr; gene-gene Pearson matrix for the augmented outcome
- `st_adata` &rarr; observed counts + spatial coordinates
- `pred_adata` &rarr; imputed expression (`pred_is_log=True`, like CellPLM)
- `U` &rarr; precomputed spatial confounders (skips SpatialPCA / R)
- `treatment` &rarr; passed as an array (no obs column needed)

In [ ]:
model = tidest(n_pcs=20, n_folds=2, corr_threshold=0.5, seed=0)
model.fit(
    sc_adata=data['sc_adata'],
    st_adata=data['st_adata'],
    pred_adata=data['pred_adata'],
    U=data['U'],
    treatment=data['A'],
    genes=data['genes'],
    pred_is_log=True,
)
model.results_.head(10)

## 3. Evaluate against the ground truth

Because this is synthetic, we know which genes are truly DE. Expect high power, a low false-positive rate, and correct effect signs.

In [ ]:
res = model.results_.copy()
is_de = dict(zip(data['genes'], data['is_de']))
res['true_de'] = res['gene'].map(is_de)
sig = res['qval'] < 0.05

n_de = int(res['true_de'].sum()); n_null = int((~res['true_de']).sum())
tpr = (sig & res['true_de']).sum() / n_de
fpr = (sig & ~res['true_de']).sum() / n_null
print(f'Detected {int(sig.sum())} / {len(res)} genes at q < 0.05')
print(f'Power (TPR): {tpr:.2f}  |  FPR: {fpr:.2f}')